# Week 18, Day 5: Stability and comparing sorts

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w018d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
people = [("Ada", 30), ("Ben", 25), ("Cy", 30), ("Dee", 25)]
print(sorted(people, key=lambda p: p[1]))

In [ ]:
def selection_sort_by(items, key):
    for i in range(len(items) - 1):
        smallest = i
        for j in range(i + 1, len(items)):
            if key(items[j]) < key(items[smallest]):
                smallest = j
        items[i], items[smallest] = items[smallest], items[i]

cards = [("hearts", 5), ("spades", 5), ("clubs", 2)]
selection_sort_by(cards, key=lambda c: c[1])
print(cards)
print(sorted([("hearts", 5), ("spades", 5), ("clubs", 2)], key=lambda c: c[1]))

*Interactive figure on the website: Watch which items move a long way in one swap (selection sort) and which only ever move past a neighbour (insertion and bubble sort). Only long jumps can carry an item past an equal one..*

In [ ]:
teams = [("Leeds", 40, 12), ("Bath", 40, 15), ("York", 38, 20), ("Hull", 40, 12)]
# points high to low, then goal difference high to low, then name A to Z
table = sorted(teams, key=lambda t: (-t[1], -t[2], t[0]))
for name, points, gd in table:
    print(f"{name:<6}{points:>4}{gd:>4}")

In [ ]:
teams = [("Leeds", 40, 12), ("Bath", 40, 15), ("York", 38, 20), ("Hull", 40, 12)]
step1 = sorted(teams, key=lambda t: t[0])                    # name A to Z
step2 = sorted(step1, key=lambda t: t[2], reverse=True)      # then goal difference
step3 = sorted(step2, key=lambda t: t[1], reverse=True)      # then points, the main key
print([t[0] for t in step3])

> **Key idea.** A stable sort keeps items that are equal by the key in their original order. Python's `sorted` and `list.sort` are always stable, so you can sort by several levels with a tuple key, or by sorting on the least important key first and the most important last.

In [ ]:
def selection_sort_by(items, key):
    for i in range(len(items) - 1):
        smallest = i
        for j in range(i + 1, len(items)):
            if key(items[j]) < key(items[smallest]):
                smallest = j
        items[i], items[smallest] = items[smallest], items[i]

cards = [("hearts", 5), ("spades", 5), ("clubs", 2)]
tagged = list(enumerate(cards))                     # (position, card)
selection_sort_by(tagged, key=lambda pc: (pc[1][1], pc[0]))
cards = [card for pos, card in tagged]
print(cards)

In [ ]:
import random

def insertion_sort(items):
    for i in range(1, len(items)):
        current = items[i]
        j = i - 1
        while j >= 0 and items[j] > current:
            items[j + 1] = items[j]
            j -= 1
        items[j + 1] = current

def check_sort(sort_function, trials=300):
    rng = random.Random(0)
    cases = [[], [1], [2, 1], [1, 2, 3], [3, 2, 1], [5, 5, 5], [2, -1, 2, 0]]
    cases += [[rng.randint(-20, 20) for _ in range(rng.randint(0, 30))] for _ in range(trials)]
    for case in cases:
        attempt = case[:]
        sort_function(attempt)
        if attempt != sorted(case):
            return f"FAILED on {case}: got {attempt}"
    return f"passed {len(cases)} cases"

print(check_sort(insertion_sort))

In [ ]:
import random, time

def selection_sort(items):
    for i in range(len(items) - 1):
        smallest = i
        for j in range(i + 1, len(items)):
            if items[j] < items[smallest]:
                smallest = j
        items[i], items[smallest] = items[smallest], items[i]

def insertion_sort(items):
    for i in range(1, len(items)):
        current = items[i]
        j = i - 1
        while j >= 0 and items[j] > current:
            items[j + 1] = items[j]
            j -= 1
        items[j + 1] = current

def bubble_sort(items):
    for end in range(len(items) - 1, 0, -1):
        swapped = False
        for j in range(end):
            if items[j] > items[j + 1]:
                items[j], items[j + 1] = items[j + 1], items[j]
                swapped = True
        if not swapped:
            break

n = 800
rng = random.Random(1)
inputs = {"random": rng.sample(range(n), n), "sorted": list(range(n)), "reversed": list(range(n, 0, -1))}
for name, sort in [("selection", selection_sort), ("insertion", insertion_sort), ("bubble", bubble_sort), ("sorted()", lambda xs: xs.sort())]:
    row = []
    for kind, data in inputs.items():
        copy = data[:]
        start = time.perf_counter()
        sort(copy)
        row.append(f"{kind} {1000 * (time.perf_counter() - start):7.1f} ms")
    print(f"{name:<10}", "  ".join(row))

### Exercise: Stable or not?

For each, set the variable to `True` if the sort is stable, `False` if not.

In [ ]:
a = ___   # sorted(items, key=...)
b = ___   # selection sort
c = ___   # insertion sort that shifts while items[j] > current
d = ___   # insertion sort that shifts while items[j] >= current
e = ___   # bubble sort that swaps when items[j] > items[j + 1]

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a is True and b is False and c is True and d is False and e is True")

### Exercise: A league table

`RESULTS` holds `(team, points, goal_difference, goals_scored)`. Write `league(results)` returning the team names ranked by points (high to low), then goal difference (high to low), then goals scored (high to low), then name (A to Z), using one `sorted` call with a tuple key.

In [ ]:
RESULTS = [("Rovers", 54, 10, 48), ("City", 54, 10, 51), ("Albion", 61, 22, 60), ("United", 54, 14, 40), ("Athletic", 54, 10, 51)]

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"ranked\"\n    assert league(RESULTS) == [\"Albion\", \"United\", \"Athletic\", \"City\", \"Rovers\"]\n    assert league([]) == []")

### Exercise: Mixed directions

Write `by_town_then_name_desc(people)` that sorts a list of `(name, town)` pairs by town A to Z, and within each town by name **Z to A**. Strings can't be negated, so use two stable sorts: first by name with `reverse=True`, then by town.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"sorted\"\n    people = [(\"Ada\", \"York\"), (\"Zed\", \"Bath\"), (\"Ben\", \"York\"), (\"Cy\", \"Bath\"), (\"Ola\", \"York\")]\n    assert by_town_then_name_desc(people) == [(\"Zed\", \"Bath\"), (\"Cy\", \"Bath\"), (\"Ola\", \"York\"), (\"Ben\", \"York\"), (\"Ada\", \"York\")]\n    assert by_town_then_name_desc([]) == []\n    assert source.count(\"sorted(\") + source.count(\".sort(\") >= 2")

### Exercise: Check stability

Write `is_stable(sort_function, records, key)` that runs `sort_function(copy, key)` on a copy of `records` (the function sorts in place by `key`) and returns `True` if the result equals `sorted(records, key=key)` and `False` otherwise. Two sort functions are in the starter; set `results` to `[is_stable(...) for each]` on the given cards.

In [ ]:
def insertion_by(items, key):
    for i in range(1, len(items)):
        current = items[i]
        j = i - 1
        while j >= 0 and key(items[j]) > key(current):
            items[j + 1] = items[j]
            j -= 1
        items[j + 1] = current

def selection_by(items, key):
    for i in range(len(items) - 1):
        smallest = i
        for j in range(i + 1, len(items)):
            if key(items[j]) < key(items[smallest]):
                smallest = j
        items[i], items[smallest] = items[smallest], items[i]

CARDS = [("hearts", 5), ("spades", 5), ("clubs", 2)]

def is_stable(sort_function, records, key):
    ___

results = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert results == [True, False]\n    assert is_stable(insertion_by, [], lambda x: x) is True")

### Exercise: Make it stable

Write `stable_selection(items, key)` that returns a **new** sorted list, sorted by `key`, using the starter's (unstable) `selection_by` but made stable with the position trick: sort `(position, item)` pairs by `(key(item), position)`, then take the items back out.

In [ ]:
def selection_by(items, key):
    for i in range(len(items) - 1):
        smallest = i
        for j in range(i + 1, len(items)):
            if key(items[j]) < key(items[smallest]):
                smallest = j
        items[i], items[smallest] = items[smallest], items[i]

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_values():\n    \"stable\"\n    cards = [(\"hearts\", 5), (\"spades\", 5), (\"clubs\", 2), (\"diamonds\", 2)]\n    assert stable_selection(cards, lambda c: c[1]) == [(\"clubs\", 2), (\"diamonds\", 2), (\"hearts\", 5), (\"spades\", 5)]\n    rng = random.Random(3)\n    for _ in range(40):\n        recs = [(rng.randint(0, 4), i) for i in range(rng.randint(0, 20))]\n        assert stable_selection(recs, lambda r: r[0]) == sorted(recs, key=lambda r: r[0])\n    assert \"sorted(\" not in source and \".sort(\" not in source")

### Bug hunt: Not quite a check

This test says the broken sort passes, because it only checks that the result is in order. The sort loses items. Improve `check` so it also catches sorts that lose, add or change items, and prints `False`.

In [ ]:
def broken_sort(items):
    items[:] = sorted(set(items))     # removes repeats!

def check(sort_function, case):
    attempt = case[:]
    sort_function(attempt)
    return all(attempt[i] <= attempt[i + 1] for i in range(len(attempt) - 1))

print(check(broken_sort, [3, 1, 3, 2]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"caught\"\n    expect_output(\"False\")\n    assert check(lambda xs: xs.sort(), [3, 1, 3, 2]) is True")

### Bug hunt: The wrong level first

This should list students by grade (A first), and alphabetically within each grade, using two stable sorts. It sorts in the wrong order of levels. Fix it so it prints `['Ama', 'Kofi', 'Bisi', 'Efe', 'Tunde']`.

In [ ]:
students = [("Tunde", "C"), ("Kofi", "A"), ("Efe", "B"), ("Ama", "A"), ("Bisi", "B")]
step1 = sorted(students, key=lambda s: s[1])
step2 = sorted(step1, key=lambda s: s[0])
print([s[0] for s in step2])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"by grade, then name\"\n    expect_output(\"['Ama', 'Kofi', 'Bisi', 'Efe', 'Tunde']\")")

### Puzzle: Randomised test

Arrange the lines so `test_sort` returns `True` only if `sort_function` sorts 200 random lists correctly. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Work these out without running them.

In [ ]:
a = ___   # sorted([("x", 2), ("y", 1), ("z", 2)], key=lambda p: p[1])
b = ___   # sorted(["bb", "a", "cc", "d"], key=len)
c = ___   # sorted([(2, "b"), (1, "z"), (2, "a")])
d = ___   # sorted(["bb", "a", "cc", "d"], key=len, reverse=True)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == [(\"y\", 1), (\"x\", 2), (\"z\", 2)], \"a\"\n    assert b == [\"a\", \"d\", \"bb\", \"cc\"], \"b\"\n    assert c == [(1, \"z\"), (2, \"a\"), (2, \"b\")], \"c: whole tuples compare item by item\"\n    assert d == [\"bb\", \"cc\", \"a\", \"d\"], \"d: reverse keeps equal items in their original order\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a sorting test bench

Write a small test bench for sorts that sort a list in place:

- `verify(sort_function, trials=200, seed=0)`: returns `None` if `sort_function` sorts correctly on the edge cases `[]`, `[1]`, `[2, 1]`, `[1, 1, 1]`, a sorted and a reversed list of 10, plus `trials` random lists (lengths 0 to 25, values 0 to 9, from `random.Random(seed)`); otherwise returns the **shortest** failing input it found (the first one, if several are equally short).
- `shrink(sort_function, case)`: given a failing case, tries removing one item at a time; whenever the shorter list still fails, keeps it and starts again; returns a case from which no single removal still fails. This is how testing tools turn a big random failure into a small one you can understand.

The checks run your bench on correct and deliberately broken sorts.

In [ ]:
import random

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def good(xs):\n    xs.sort()\ndef drops_last(xs):\n    xs.sort()\n    if len(xs) > 3:\n        xs.pop()\ndef wrong_on_repeats(xs):\n    xs[:] = sorted(set(xs))\ndef one_pass_bubble(xs):\n    for j in range(len(xs) - 1):\n        if xs[j] > xs[j + 1]:\n            xs[j], xs[j + 1] = xs[j + 1], xs[j]\ndef test_verify():\n    \"verify\"\n    assert verify(good) is None\n    bad = verify(drops_last)\n    assert bad is not None and len(bad) == 4\n    r = verify(wrong_on_repeats)\n    assert r is not None and len(set(r)) < len(r) and len(r) <= 3\n    assert verify(one_pass_bubble) is not None\ndef test_shrink():\n    \"shrink\"\n    s = shrink(one_pass_bubble, [5, 9, 1, 7, 3, 8, 2])\n    xs = s[:]\n    one_pass_bubble(xs)\n    assert xs != sorted(s)\n    for i in range(len(s)):\n        smaller = s[:i] + s[i + 1:]\n        ys = smaller[:]\n        one_pass_bubble(ys)\n        assert ys == sorted(smaller)")

## Warm-up

Ben and Dee are both 25; Ada and Cy are both 30. After sorting by age, Ben still comes before Dee, and Ada before Cy: the same order they had to begin with. That isn't luck. Python's sort promises it.

## What stability means

A sort is **stable** if items that are **equal by the sort key** stay in the same order relative to each other as they were before sorting.

Sorting plain numbers, stability makes no visible difference: one 7 looks like any other. It matters when you sort **records by one field**, because records with the same value in that field can still be different records.

| Sort | Stable? | Why |
|---|---|---|
| `sorted`, `list.sort` | yes, guaranteed | Timsort is designed to be (Week 19) |
| insertion sort | yes | an item never moves past an equal one (`>` not `>=`) |
| bubble sort | yes | only swaps neighbours that are strictly out of order |
| selection sort | no | its long-distance swap can jump an item over an equal one |
| merge sort | yes, if the merge takes from the left on ties | Week 19 |
| quicksort | usually no | Week 19 |

Here's selection sort breaking the order of equal items:

The first swap sends `("hearts", 5)` from the front to the back, past `("spades", 5)`. Both results are correctly sorted by number, but only `sorted` kept hearts before spades.

## Sorting by several things

Real sorting is often by several levels: a league table by points, then by goal difference, then by name. There are two ways.

**A tuple key.** Tuples compare item by item, first by their first item, then by the second if the first is equal, and so on. So a key that returns a tuple sorts by several levels at once:

Negating a number reverses its order, which is how you mix "high to low" and "A to Z" in one key. (Strings can't be negated; for those, use the next method.)

**Repeated stable sorts.** Sort by the **least** important key first, then by the next, ending with the most important. Each later sort is stable, so it keeps the order the earlier ones made among items it sees as equal:

Same answer. This method only works because each sort is stable, and it's the right tool when the levels go in different directions and can't be negated, such as names Z to A within points high to low. (`reverse=True` keeps stability too: equal items stay in their original order.)

## Making a sort stable

Any sort can be made stable by adding the item's **original position** to its key as a tie-breaker. Two items can then never be equal, so there's nothing for the sort to reorder:

This trick ("decorate, sort, undecorate") costs O(n) extra space for the positions.

## Testing a sort

A sort is easy to get **nearly** right. Checking it on one example isn't enough. A good test checks two things:

1. **The result is in order:** every item is less than or equal to the next.
2. **The result has exactly the same items:** nothing lost, nothing added, nothing duplicated. Comparing with `sorted(original)` checks both at once.

And it tries the cases where sorts go wrong: an empty list, one item, two items, already sorted, reversed, all equal, repeats, negative numbers, and plenty of random lists.

This style, where you test against a simple trusted answer on lots of random inputs, is called **randomised testing** or **property testing**, and it finds bugs that hand-picked examples miss. You'll use the `hypothesis` library to do it properly in Phase 2.

To test **stability**, sort records whose keys repeat, and check that items with equal keys keep their original order. Comparing with `sorted(..., key=...)`, which is stable, does exactly that.

## Comparing sorts

To compare sorts fairly, measure them on the **same data**, at **several sizes**, on **several kinds** of input:

Read the table, then explain it with this week's analysis:

- **Selection sort** takes about the same time on all three: always n(n − 1)/2 comparisons.
- **Insertion and bubble sort** are near-instant on sorted data, O(n), and slowest on reversed data, the most inversions.
- **Bubble sort** loses to insertion sort on random data: the same number of moves (one per inversion), but each is a full swap, and it compares more.
- **`sorted`** is hundreds of times faster than all of them: O(n log n), and written in C. In real code, always use it.

So why learn the others? Because they're the clearest way to learn to think about algorithms, because insertion sort lives on inside Timsort, and because writing a sort by hand is a classic interview question.

## Practice

## Quiz

## Daily challenge

## Before you go

Sort your phone contacts, music or files by two levels in your head (say, by artist and then by year), and say which level a stable sort must do first. Tomorrow is the Forge: you'll build a small sorting and searching toolkit and race it against Python's own.